In [1]:
import os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW = os.path.join(PROJECT_ROOT, "data/raw")
PROC = os.path.join(PROJECT_ROOT, "data/processed")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
print("Project root:", PROJECT_ROOT)

Project root: f:\BRACU\CSE425\Project\gnn-bert-music-context


In [2]:
print(os.getcwd())

f:\BRACU\CSE425\Project\gnn-bert-music-context


1a: MagnaTagATune

In [3]:
import urllib.request

os.makedirs(RAW, exist_ok=True)
url = "https://huggingface.co/datasets/confit/magnatagatune/resolve/main/annotations_final.csv"
dest = os.path.join(RAW, "mtat_annotations.csv")

if not os.path.exists(dest):
    urllib.request.urlretrieve(url, dest)

print(os.path.getsize(dest), "bytes")

21517373 bytes


In [4]:
import pandas as pd

mtat = pd.read_csv(os.path.join(RAW, "mtat_annotations.csv"), sep="\t")
print(mtat.shape)

tag_cols = mtat.columns[1:-1]
top50 = mtat[tag_cols].sum().sort_values(ascending=False).head(50).index.tolist()
mtat_top50 = mtat[["clip_id"] + top50 + ["mp3_path"]]
mtat_top50.to_csv(os.path.join(PROC, "mtat_top50_tags.csv"), index=False)
print(mtat_top50.shape)

(25863, 190)
(25863, 52)


1b: FMA-small

In [ ]:
FMA_DIR = os.path.join(RAW, "fma")
os.makedirs(FMA_DIR, exist_ok=True)

meta_url = "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip"
meta_zip = os.path.join(FMA_DIR, "fma_metadata.zip")
if not os.path.exists(meta_zip):
    print("Downloading FMA metadata...")
    urllib.request.urlretrieve(meta_url, meta_zip)

print(os.path.getsize(meta_zip), "bytes")

358412441 bytes


In [ ]:
import zipfile

with zipfile.ZipFile(meta_zip, 'r') as z:
    z.extractall(FMA_DIR)

print(os.listdir(os.path.join(FMA_DIR, "fma_metadata")))

In [ ]:
import requests
from tqdm import tqdm

audio_url = "https://os.unil.cloud.switch.ch/fma/fma_small.zip"
audio_zip = os.path.join(FMA_DIR, "fma_small.zip")

if not os.path.exists(audio_zip):
    response = requests.get(audio_url, stream=True)
    total_size = int(response.headers.get('content-length', 0))

    with open(audio_zip, 'wb') as f, tqdm(
        total=total_size, unit='B', unit_scale=True, unit_divisor=1024, desc="FMA-small audio"
    ) as pbar:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            pbar.update(len(chunk))

print(os.path.getsize(audio_zip), "bytes") 

7679594875 bytes


In [ ]:
import zipfile

with zipfile.ZipFile(audio_zip, 'r') as z:
    z.extractall(FMA_DIR)

mp3_files = []
for root, dirs, files in os.walk(FMA_DIR):
    for f in files:
        if f.endswith(".mp3"):
            mp3_files.append(os.path.join(root, f))

print("Total mp3 files:", len(mp3_files))

Total mp3 files: 8000


In [ ]:
tracks = pd.read_csv(f"{FMA_DIR}/fma_metadata/tracks.csv", index_col=0, header=[0, 1])
small = tracks[tracks[('set', 'subset')] == 'small']
print(small.shape) 

small_meta = small[('track', 'genre_top')].reset_index()
small_meta.columns = ['track_id', 'genre']
small_meta.to_csv(os.path.join(PROC, "fma_small_labels.csv"), index=False)
print(small_meta.shape)

(8000, 52)
(8000, 2)


1c — MusicCaps captions CSV

In [ ]:
MC_DIR = os.path.join(RAW, "musiccaps")
os.makedirs(MC_DIR, exist_ok=True)

mc_url = "https://huggingface.co/datasets/google/MusicCaps/resolve/main/musiccaps-public.csv"
mc_path = os.path.join(MC_DIR, "musiccaps-public.csv")

if not os.path.exists(mc_path):
    urllib.request.urlretrieve(mc_url, mc_path)

mc = pd.read_csv(mc_path)
print(mc.shape) 

(5521, 9)


In [ ]:
%pip install -q -U yt-dlp yt-dlp-ejs
!deno --version

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


deno 2.9.6 (stable, release, x86_64-pc-windows-msvc)
v8 15.0.245.2-rusty
typescript 6.0.3


In [9]:
cookies_path = os.path.join(PROJECT_ROOT, "cookies.txt")
print("Cookies file present:", os.path.exists(cookies_path))

Cookies file present: True


In [10]:
import subprocess

AUDIO_DIR = os.path.join(MC_DIR, "audio")
os.makedirs(AUDIO_DIR, exist_ok=True)

test_row = mc.iloc[0]
test_id, start_s, end_s = test_row["ytid"], test_row["start_s"], test_row["end_s"]
out_path = os.path.join(AUDIO_DIR, f"{test_id}.wav")

result = subprocess.run(
    ["yt-dlp", "--cookies", cookies_path,
     "--remote-components", "ejs:github",
     "--download-sections", f"*{start_s}-{end_s}",
     "--force-keyframes-at-cuts",
     "-x", "--audio-format", "wav",
     "-o", out_path,
     f"https://www.youtube.com/watch?v={test_id}"],
    capture_output=True, text=True, timeout=60
)
print("Return code:", result.returncode)
print(result.stdout[-500:])
print(result.stderr[-500:])

Return code: 0
nloading 1 format(s): 251
[info] -0Gj8-vB1q4: Downloading 1 time ranges: 30.0-40.0
[download] Destination: f:\BRACU\CSE425\Project\gnn-bert-music-context\data\raw\musiccaps\audio\-0Gj8-vB1q4.webm

[download] 100% of  124.69KiB in 00:00:00 at 469.54KiB/s
[ExtractAudio] Destination: f:\BRACU\CSE425\Project\gnn-bert-music-context\data\raw\musiccaps\audio\-0Gj8-vB1q4.wav
Deleting original file f:\BRACU\CSE425\Project\gnn-bert-music-context\data\raw\musiccaps\audio\-0Gj8-vB1q4.webm (pass -k to keep)

BRACU\CSE425\Project\gnn-bert-music-context\data\raw\musiccaps\audio\-0Gj8-vB1q4.webm.part':
  Metadata:
    encoder         : Lavf63.1.101
  Stream #0:0(eng): Audio: opus, 48000 Hz, stereo, flt, 96 kb/s (default)
    Metadata:
      encoder         : Lavc63.1.101 libopus
[out#0/webm @ 0000026925ebb700] video:0KiB audio:121KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 3.330096%
size=     125KiB time=00:00:10.00 bitrate= 102.1kbits/s speed=85.9x elapsed

In [11]:
import subprocess
result = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1", out_path],
    capture_output=True, text=True
)
print(result.stdout)

duration=10.000000



In [ ]:
import time

MANIFEST_PATH = os.path.join(PROC, "musiccaps_download_manifest.csv")

if os.path.exists(MANIFEST_PATH):
    manifest = pd.read_csv(MANIFEST_PATH)
else:
    manifest = pd.DataFrame(columns=["ytid", "success", "error"])

done_ids = set(manifest["ytid"])
print(f"Already attempted: {len(done_ids)}")

def download_clip(ytid, start_s, end_s):
    out_path = os.path.join(AUDIO_DIR, f"{ytid}.wav")
    if os.path.exists(out_path):
        os.remove(out_path)
    result = subprocess.run(
        ["yt-dlp", "--cookies", cookies_path,
         "--remote-components", "ejs:github",
         "--download-sections", f"*{start_s}-{end_s}",
         "--force-keyframes-at-cuts",
         "-x", "--audio-format", "wav",
         "-o", out_path,
         f"https://www.youtube.com/watch?v={ytid}"],
        capture_output=True, text=True, timeout=60
    )
    return result.returncode == 0, (result.stderr[-200:] if result.returncode != 0 else "")

TARGET_ATTEMPTS = 300
attempted = 0
new_rows = []

for _, row in tqdm(mc.iterrows(), total=len(mc)):
    if attempted >= TARGET_ATTEMPTS:
        break
    if row["ytid"] in done_ids:
        continue
    success, error = download_clip(row["ytid"], row["start_s"], row["end_s"])
    new_rows.append({"ytid": row["ytid"], "success": success, "error": error})
    done_ids.add(row["ytid"])
    attempted += 1
    time.sleep(1)

    if attempted % 25 == 0:
        pd.concat([manifest, pd.DataFrame(new_rows)]).to_csv(MANIFEST_PATH, index=False)

manifest = pd.concat([manifest, pd.DataFrame(new_rows)])
manifest.to_csv(MANIFEST_PATH, index=False)
print(manifest["success"].value_counts())

Already attempted: 900


  0%|          | 0/5521 [00:00<?, ?it/s]

 22%|██▏       | 1200/5521 [27:37<1:39:28,  1.38s/it] 

success
True     1159
False      41
Name: count, dtype: int64
